In [1]:
## 1. Initialize the Baseline Training Environment
!git clone https://github.com/Raghavtripathii/docshield-ai.git
%cd /content/docshield-ai
!python -m pip install --quiet -r requirements-dev.txt

Cloning into 'docshield-ai'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 119 (delta 51), reused 83 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 744.88 KiB | 26.60 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/docshield-ai
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
## 2. Verify T4 GPU
import torch

assert torch.cuda.is_available()

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

torch.cuda.empty_cache()

PyTorch: 2.11.0+cu128
GPU: Tesla T4


In [3]:
## 3. Establish Reproducibility
from src.config import CONFIG
from src.reproducibility import set_seed

set_seed(CONFIG.seed)

print("Seed:", CONFIG.seed)

Seed: 42


In [4]:
## 4. Build Training and Evaluation Datasets
from src.data import load_funsd
from src.dataset import FUNSDDataset
from src.labels import get_label_list
from src.processor import build_processor

dataset = load_funsd()
processor = build_processor()
label_list = get_label_list(dataset)

train_dataset = FUNSDDataset(
    dataset["train"],
    processor,
)

eval_dataset = FUNSDDataset(
    dataset["test"],
    processor,
)

print("Train:", len(train_dataset))
print("Evaluation:", len(eval_dataset))
print("Labels:", len(label_list))

README.md:   0%|          | 0.00/755 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 12.3MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.38MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/856 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

Train: 149
Evaluation: 50
Labels: 7


In [6]:
## 5. Construct the Baseline LayoutLMv3 Model
from src.model import build_model
from src.model_stats import count_parameters

model = build_model(label_list)

parameter_stats = count_parameters(model)

print(parameter_stats)

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'total': 125332359, 'trainable': 125332359, 'frozen': 0, 'trainable_percentage': 100.0}


In [7]:
## 6. Validate a Forward Pass
sample = train_dataset[0]

batch = {
    key: value.unsqueeze(0).to("cuda")
    for key, value in sample.items()
}

model = model.to("cuda")

with torch.no_grad():
    output = model(**batch)

print("Loss:", float(output.loss))
print("Logits:", tuple(output.logits.shape))

assert torch.isfinite(output.loss)

del batch
torch.cuda.empty_cache()

print("Forward pass: PASSED")


Loss: 2.4049551486968994
Logits: (1, 512, 7)
Forward pass: PASSED


In [8]:
## 7. Verify the Installed Transformers Training API

import inspect
import transformers
from transformers import Trainer, TrainingArguments

print("Transformers version:", transformers.__version__)
print("TrainingArguments module:", TrainingArguments.__module__)

signature = inspect.signature(TrainingArguments.__init__)

required_arguments = [
    "output_dir",
    "eval_strategy",
    "per_device_train_batch_size",
    "gradient_accumulation_steps",
    "fp16",
]

for argument in required_arguments:
    print(
        f"{argument}:",
        "SUPPORTED" if argument in signature.parameters else "MISSING"
    )

assert all(
    argument in signature.parameters
    for argument in required_arguments
)

print("Training API validation: PASSED")

Transformers version: 5.13.1
TrainingArguments module: transformers.training_args
output_dir: SUPPORTED
eval_strategy: SUPPORTED
per_device_train_batch_size: SUPPORTED
gradient_accumulation_steps: SUPPORTED
fp16: SUPPORTED
Training API validation: PASSED


In [9]:
## 8. Build Entity-Level Evaluation Metrics

from src.metrics import (
    build_seqeval_metric,
    compute_seqeval_metrics,
)

seqeval_metric = build_seqeval_metric()

def compute_metrics(eval_prediction):
    predictions = eval_prediction.predictions
    labels = eval_prediction.label_ids

    return compute_seqeval_metrics(
        predictions=predictions,
        labels=labels,
        label_list=label_list,
        metric=seqeval_metric,
    )

print("Evaluation metrics: READY")
print("Primary metric: entity-level F1")

Evaluation metrics: READY
Primary metric: entity-level F1


In [10]:
## 9. Validate the Metric Pipeline with Synthetic Predictions

import numpy as np

synthetic_labels = np.array([
    [0, 1, 2, -100, -100]
])

synthetic_logits = np.full(
    (1, 5, len(label_list)),
    -10.0,
    dtype=np.float32,
)

synthetic_logits[0, 0, 0] = 10.0
synthetic_logits[0, 1, 1] = 10.0
synthetic_logits[0, 2, 2] = 10.0

synthetic_metrics = compute_seqeval_metrics(
    predictions=synthetic_logits,
    labels=synthetic_labels,
    label_list=label_list,
    metric=seqeval_metric,
)

print(synthetic_metrics)

assert all(
    key in synthetic_metrics
    for key in [
        "precision",
        "recall",
        "f1",
        "accuracy",
    ]
)

print("Metric pipeline: PASSED")

{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'accuracy': 1.0}
Metric pipeline: PASSED


In [11]:
## 10. Load the Versioned Baseline Experiment Configuration

import json
from pathlib import Path

CONFIG_PATH = Path("configs/baseline.json")

with CONFIG_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    baseline_config = json.load(file)

for key, value in baseline_config.items():
    print(f"{key}: {value}")

assert baseline_config["model_name"] == "microsoft/layoutlmv3-base"
assert baseline_config["seed"] == 42
assert baseline_config["max_length"] == 512

print("Baseline configuration: VALIDATED")

experiment_name: layoutlmv3_funsd_baseline
model_name: microsoft/layoutlmv3-base
dataset_name: nielsr/funsd
seed: 42
max_length: 512
learning_rate: 5e-05
train_batch_size: 2
eval_batch_size: 2
gradient_accumulation_steps: 2
num_train_epochs: 3
weight_decay: 0.01
warmup_ratio: 0.1
fp16: True
save_strategy: epoch
logging_steps: 10
Baseline configuration: VALIDATED


In [12]:
## 11. Configure Memory-Conscious LayoutLMv3 Fine-Tuning

from transformers import TrainingArguments

OUTPUT_DIR = "/content/docshield-baseline"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    learning_rate=baseline_config["learning_rate"],

    per_device_train_batch_size=baseline_config[
        "train_batch_size"
    ],

    per_device_eval_batch_size=baseline_config[
        "eval_batch_size"
    ],

    gradient_accumulation_steps=baseline_config[
        "gradient_accumulation_steps"
    ],

    num_train_epochs=baseline_config[
        "num_train_epochs"
    ],

    weight_decay=baseline_config["weight_decay"],
    warmup_ratio=baseline_config["warmup_ratio"],

    fp16=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=baseline_config["logging_steps"],

    eval_accumulation_steps=1,

    save_total_limit=1,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    seed=baseline_config["seed"],
    data_seed=baseline_config["seed"],

    report_to="none",

    remove_unused_columns=False,
)

print("Output directory:", OUTPUT_DIR)
print(
    "Train batch size:",
    training_args.per_device_train_batch_size,
)
print(
    "Gradient accumulation:",
    training_args.gradient_accumulation_steps,
)
print(
    "Epochs:",
    training_args.num_train_epochs,
)
print("FP16:", training_args.fp16)
print("Evaluation:", training_args.eval_strategy)

print("Training configuration: READY")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Output directory: /content/docshield-baseline
Train batch size: 2
Gradient accumulation: 2
Epochs: 3
FP16: True
Evaluation: IntervalStrategy.EPOCH
Training configuration: READY


In [13]:
## 12. Construct the LayoutLMv3 Trainer

from transformers import Trainer, default_data_collator

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,

    data_collator=default_data_collator,

    processing_class=processor,

    compute_metrics=compute_metrics,
)

print("Trainer:", trainer.__class__.__name__)
print("Training samples:", len(train_dataset))
print("Evaluation samples:", len(eval_dataset))

print("Trainer construction: PASSED")

Trainer: Trainer
Training samples: 149
Evaluation samples: 50
Trainer construction: PASSED


In [14]:
## 13. Inspect T4 Memory Before Training

import torch

torch.cuda.empty_cache()

device_index = torch.cuda.current_device()

total_memory = (
    torch.cuda.get_device_properties(device_index).total_memory
    / 1024**3
)

allocated_memory = (
    torch.cuda.memory_allocated(device_index)
    / 1024**3
)

reserved_memory = (
    torch.cuda.memory_reserved(device_index)
    / 1024**3
)

print("GPU:", torch.cuda.get_device_name(device_index))
print(f"Total VRAM:     {total_memory:.2f} GB")
print(f"Allocated VRAM: {allocated_memory:.2f} GB")
print(f"Reserved VRAM:  {reserved_memory:.2f} GB")

print("GPU memory inspection: COMPLETE")

GPU: Tesla T4
Total VRAM:     14.56 GB
Allocated VRAM: 0.48 GB
Reserved VRAM:  0.52 GB
GPU memory inspection: COMPLETE


In [15]:
## 14. Validate the Complete Evaluation Pipeline Before Training

pretraining_metrics = trainer.evaluate()

print("\nPre-training evaluation:")

for key, value in pretraining_metrics.items():
    print(f"{key}: {value}")

required_metric_keys = [
    "eval_precision",
    "eval_recall",
    "eval_f1",
    "eval_accuracy",
]

for key in required_metric_keys:
    assert key in pretraining_metrics

print("\nPre-training evaluation pipeline: PASSED")

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
No log,2.130226,0,0.016906,0.023315,0.019599,0.069770



Pre-training evaluation:
eval_loss: 2.1302261352539062
eval_precision: 0.016905549430356485
eval_recall: 0.02331474911302585
eval_f1: 0.019599488708990198
eval_accuracy: 0.06977022498803255

Pre-training evaluation pipeline: PASSED


In [16]:
## 15. Record the Baseline Training Environment

import platform
import time

experiment_environment = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "cuda_available": torch.cuda.is_available(),
    "train_documents": len(train_dataset),
    "evaluation_documents": len(eval_dataset),
    "num_labels": len(label_list),
    "seed": baseline_config["seed"],
}

for key, value in experiment_environment.items():
    print(f"{key}: {value}")

python: 3.12.13
torch: 2.11.0+cu128
transformers: 5.13.1
gpu: Tesla T4
cuda_available: True
train_documents: 149
evaluation_documents: 50
num_labels: 7
seed: 42


In [17]:
## 16. Fine-Tune LayoutLMv3 on FUNSD

import time

torch.cuda.empty_cache()

training_start = time.perf_counter()

train_result = trainer.train()

training_end = time.perf_counter()

training_time_seconds = (
    training_end - training_start
)

print("\n" + "=" * 60)
print("BASELINE TRAINING COMPLETE")
print("=" * 60)

print(
    f"Training time: "
    f"{training_time_seconds:.2f} seconds"
)

print(
    f"Training loss: "
    f"{train_result.training_loss:.6f}"
)

print("=" * 60)

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,2.196980,0.810236,0.559025,0.662443,0.606356,0.726424
2,1.512846,0.619213,0.704925,0.790674,0.745342,0.781953
3,0.934211,0.581392,0.738440,0.801318,0.768595,0.792963


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


BASELINE TRAINING COMPLETE
Training time: 141.28 seconds
Training loss: 1.758415


In [18]:
## 17. Evaluate the Fine-Tuned Baseline Model

final_metrics = trainer.evaluate()

print("\n" + "=" * 60)
print("FINAL BASELINE EVALUATION")
print("=" * 60)

for key, value in final_metrics.items():
    print(f"{key}: {value}")

print("=" * 60)

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.934211,0.581392,3,0.738440,0.801318,0.768595,0.792963



FINAL BASELINE EVALUATION
eval_loss: 0.5813916325569153
eval_precision: 0.7384399813171415
eval_recall: 0.801317790167258
eval_f1: 0.768595041322314
eval_accuracy: 0.7929631402584969


In [19]:
## 18. Compare Pre-Training and Fine-Tuned Performance

comparison = {
    "precision": (
        pretraining_metrics["eval_precision"],
        final_metrics["eval_precision"],
    ),
    "recall": (
        pretraining_metrics["eval_recall"],
        final_metrics["eval_recall"],
    ),
    "f1": (
        pretraining_metrics["eval_f1"],
        final_metrics["eval_f1"],
    ),
    "accuracy": (
        pretraining_metrics["eval_accuracy"],
        final_metrics["eval_accuracy"],
    ),
}

print(
    f"{'Metric':<12}"
    f"{'Before':>12}"
    f"{'After':>12}"
    f"{'Change':>12}"
)

print("-" * 48)

for metric, (before, after) in comparison.items():
    print(
        f"{metric:<12}"
        f"{before:>12.4f}"
        f"{after:>12.4f}"
        f"{after-before:>+12.4f}"
    )

Metric            Before       After      Change
------------------------------------------------
precision         0.0169      0.7384     +0.7215
recall            0.0233      0.8013     +0.7780
f1                0.0196      0.7686     +0.7490
accuracy          0.0698      0.7930     +0.7232


In [20]:
## 19. Validate the Final Baseline Measurements

import math

required_results = {
    "precision": final_metrics["eval_precision"],
    "recall": final_metrics["eval_recall"],
    "f1": final_metrics["eval_f1"],
    "accuracy": final_metrics["eval_accuracy"],
}

for name, value in required_results.items():
    assert math.isfinite(value)
    assert 0.0 <= value <= 1.0

    print(
        f"{name:10s}: "
        f"{value:.6f}  VALID"
    )

assert math.isfinite(
    final_metrics["eval_loss"]
)

print("\nFinal baseline measurements: VALIDATED")

precision : 0.738440  VALID
recall    : 0.801318  VALID
f1        : 0.768595  VALID
accuracy  : 0.792963  VALID

Final baseline measurements: VALIDATED


In [21]:
## 20. Save Genuine Baseline Experiment Metrics

import json
from pathlib import Path

results_directory = Path("results")
results_directory.mkdir(
    parents=True,
    exist_ok=True,
)

baseline_results = {
    "experiment": baseline_config[
        "experiment_name"
    ],

    "model": baseline_config[
        "model_name"
    ],

    "dataset": baseline_config[
        "dataset_name"
    ],

    "seed": baseline_config["seed"],

    "precision": float(
        final_metrics["eval_precision"]
    ),

    "recall": float(
        final_metrics["eval_recall"]
    ),

    "f1": float(
        final_metrics["eval_f1"]
    ),

    "accuracy": float(
        final_metrics["eval_accuracy"]
    ),

    "eval_loss": float(
        final_metrics["eval_loss"]
    ),

    "training_loss": float(
        train_result.training_loss
    ),

    "training_time_seconds": float(
        training_time_seconds
    ),

    "gpu": torch.cuda.get_device_name(0),

    "train_documents": len(train_dataset),

    "evaluation_documents": len(
        eval_dataset
    ),
}

RESULT_PATH = (
    results_directory
    / "baseline_metrics.json"
)

with RESULT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        baseline_results,
        file,
        indent=2,
    )

print(
    json.dumps(
        baseline_results,
        indent=2,
    )
)

print(
    "\nSaved:",
    RESULT_PATH,
)

{
  "experiment": "layoutlmv3_funsd_baseline",
  "model": "microsoft/layoutlmv3-base",
  "dataset": "nielsr/funsd",
  "seed": 42,
  "precision": 0.7384399813171415,
  "recall": 0.801317790167258,
  "f1": 0.768595041322314,
  "accuracy": 0.7929631402584969,
  "eval_loss": 0.5813916325569153,
  "training_loss": 1.7584147411480284,
  "training_time_seconds": 141.27777706300003,
  "gpu": "Tesla T4",
  "train_documents": 149,
  "evaluation_documents": 50
}

Saved: results/baseline_metrics.json


In [22]:
## 21. Save the Best Fine-Tuned LayoutLMv3 Checkpoint

FINAL_MODEL_DIR = (
    "/content/docshield-layoutlmv3-baseline"
)

trainer.save_model(
    FINAL_MODEL_DIR
)

processor.save_pretrained(
    FINAL_MODEL_DIR
)

print(
    "Model saved to:",
    FINAL_MODEL_DIR,
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/docshield-layoutlmv3-baseline


In [23]:
## 22. Inspect the Training and Validation History

history = trainer.state.log_history

print(
    "Logged events:",
    len(history)
)

for event in history:
    print(event)

Logged events: 16
{'loss': 3.903137969970703, 'grad_norm': 12.694794654846191, 'learning_rate': 3.7500000000000003e-05, 'epoch': 0.26666666666666666, 'step': 10}
{'loss': 2.8299333572387697, 'grad_norm': 27.538841247558594, 'learning_rate': 4.656862745098039e-05, 'epoch': 0.5333333333333333, 'step': 20}
{'loss': 2.196980094909668, 'grad_norm': 20.94991683959961, 'learning_rate': 4.166666666666667e-05, 'epoch': 0.8, 'step': 30}
{'eval_loss': 0.8102355003356934, 'eval_model_preparation_time': 0.0038, 'eval_precision': 0.5590248075278016, 'eval_recall': 0.6624429802331475, 'eval_f1': 0.6063558339132453, 'eval_accuracy': 0.7264241263762565, 'eval_runtime': 3.5898, 'eval_samples_per_second': 13.928, 'eval_steps_per_second': 6.964, 'epoch': 1.0, 'step': 38}
{'loss': 1.8607587814331055, 'grad_norm': 13.986846923828125, 'learning_rate': 3.6764705882352945e-05, 'epoch': 1.0533333333333332, 'step': 40}
{'loss': 1.4868938446044921, 'grad_norm': 35.407047271728516, 'learning_rate': 3.1862745098039

In [24]:
## 23. Identify the Best Baseline Checkpoint

print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint,
)

print(
    "Best metric:",
    trainer.state.best_metric,
)

print(
    "Global training steps:",
    trainer.state.global_step,
)

assert trainer.state.best_metric is not None

print("Best-model selection: PASSED")

Best checkpoint: /content/docshield-baseline/checkpoint-114
Best metric: 0.768595041322314
Global training steps: 114
Best-model selection: PASSED


In [25]:
## 24. Run the Complete DocShield Test Suite After Training

!python -m pytest -q

.............................                                            [100%]
29 passed in 62.66s (0:01:02)


In [26]:
## 25. Baseline Experiment Summary

print("=" * 68)
print("DOCSHIELD — LAYOUTLMV3 CLEAN-DOCUMENT BASELINE")
print("=" * 68)

print(
    "Model:",
    baseline_config["model_name"],
)

print("Dataset: FUNSD")

print(
    "Training documents:",
    len(train_dataset),
)

print(
    "Evaluation documents:",
    len(eval_dataset),
)

print(
    "Parameters:",
    f"{parameter_stats['total']:,}",
)

print(
    "Trainable parameters:",
    f"{parameter_stats['trainable']:,}",
)

print(
    "Precision:",
    f"{final_metrics['eval_precision']:.4f}",
)

print(
    "Recall:",
    f"{final_metrics['eval_recall']:.4f}",
)

print(
    "F1:",
    f"{final_metrics['eval_f1']:.4f}",
)

print(
    "Accuracy:",
    f"{final_metrics['eval_accuracy']:.4f}",
)

print(
    "Evaluation loss:",
    f"{final_metrics['eval_loss']:.4f}",
)

print(
    "Training time:",
    f"{training_time_seconds:.2f} seconds",
)

print(
    "GPU:",
    torch.cuda.get_device_name(0),
)

print("=" * 68)
print("BASELINE EXPERIMENT: COMPLETE")
print("=" * 68)

DOCSHIELD — LAYOUTLMV3 CLEAN-DOCUMENT BASELINE
Model: microsoft/layoutlmv3-base
Dataset: FUNSD
Training documents: 149
Evaluation documents: 50
Parameters: 125,332,359
Trainable parameters: 125,332,359
Precision: 0.7384
Recall: 0.8013
F1: 0.7686
Accuracy: 0.7930
Evaluation loss: 0.5814
Training time: 141.28 seconds
GPU: Tesla T4
BASELINE EXPERIMENT: COMPLETE
